In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
import torch
import networkx as nx
import matplotlib.pyplot as plt
import random
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import shutil
import subprocess

REPO_URL = "https://github.com/samuncleML/Bot-Detection.git"
CLONE_DIR = "Bot-Detection"

# Delete the path if it already exists
if os.path.exists(CLONE_DIR):
    print(f"🗑️ Directory '{CLONE_DIR}' exists. Deleting it...")
    shutil.rmtree(CLONE_DIR)

# Clone the repository freshly
print(f"🔄 Cloning {REPO_URL}...")
subprocess.run(["git", "clone", REPO_URL, CLONE_DIR])

🔄 Cloning https://github.com/samuncleML/Bot-Detection.git...


Cloning into 'Bot-Detection'...


In [ ]:
!pip install --upgrade torchao -q
!pip install trl --no-deps -q
!pip install -U bitsandbytes>=0.46.1 -q

In [ ]:
import os
import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
GRPO_CHECKPOINT = "/kaggle/working/Bot-Detection/qwen_graphmind_grpo_final"

# 1. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load base model in 4-bit — makes 2048 seq len fit on T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False

# 3. Load GRPO adapter as starting point
model = PeftModel.from_pretrained(base_model, GRPO_CHECKPOINT, is_trainable=True)
model.print_trainable_parameters()

# 4. Load new clean dataset
jsonl_path = "/kaggle/working/Bot-Detection/data/qwen_combined_train_v2.jsonl"
raw_dataset = load_dataset("json", data_files=jsonl_path, split="train")

def preprocess_function(examples):
    texts = [
        tokenizer.apply_chat_template(
            msg, tokenize=False, add_generation_prompt=False
        )
        for msg in examples["messages"]
    ]
    return tokenizer(texts, max_length=2048, truncation=True, padding=False)

tokenized_dataset = raw_dataset.map(
    preprocess_function, batched=True, remove_columns=raw_dataset.column_names
)

# 5. Training arguments
is_bf16 = torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen_graphmind_sft_v2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    num_train_epochs=1,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    logging_steps=20,
    save_strategy="epoch",
    fp16=not is_bf16,
    bf16=is_bf16,
    optim="paged_adamw_8bit",       # memory-efficient optimizer for 4-bit training
    seed=42,
    report_to="none",
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    ddp_find_unused_parameters=False,
)

# 6. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("🚀 Continuing SFT from GRPO checkpoint with max_length=2048...")
trainer.train()

# 7. Save
trainer.model.save_pretrained("/kaggle/working/Bot-Detection/qwen_graphmind_sft_v2_final")
tokenizer.save_pretrained("/kaggle/working/Bot-Detection/qwen_graphmind_sft_v2_final")
print("✅ Done! Saved to '/kaggle/working/Bot-Detection/qwen_graphmind_sft_v2_final'.")

# 8. Calculate final loss
print("\n--- Training Loss Summary ---")
log_history = trainer.state.log_history

train_losses = [(l["step"], l["loss"]) for l in log_history if "loss" in l]
eval_losses  = [(l["step"], l["eval_loss"]) for l in log_history if "eval_loss" in l]

if train_losses:
    steps, losses = zip(*train_losses)
    print(f"  Initial loss  (step {steps[0]:>4}): {losses[0]:.4f}")
    print(f"  Final loss    (step {steps[-1]:>4}): {losses[-1]:.4f}")
    print(f"  Improvement  : {losses[0] - losses[-1]:.4f} ({((losses[0]-losses[-1])/losses[0])*100:.1f}%)")

    import torch, math
    final_perplexity = math.exp(losses[-1])
    print(f"  Perplexity   : {final_perplexity:.4f}")

print("\n  Full loss curve:")
for step, loss in train_losses:
    print(f"    step {step:>4} | loss: {loss:.4f}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

🚀 Continuing SFT from GRPO checkpoint with max_length=2048...


Step,Training Loss


In [ ]:
'''import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
  
CHECKPOINT = "/kaggle/working/Bot-Detection/checkpoints/checkpoint-1500"
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Model & Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map=None,
    trust_remote_code=True,
).to("cuda")

# Load the LoRA adapter weights from checkpoint instead of fresh LoRA
model = PeftModel.from_pretrained(base_model, CHECKPOINT, is_trainable=True)
model.print_trainable_parameters()

# 3. Load & Preprocess JSONL Dataset
jsonl_path = "/kaggle/working/Bot-Detection/data/qwen_combined_train.jsonl"
raw_dataset = load_dataset("json", data_files=jsonl_path, split="train")

def preprocess_function(examples):
    texts = [
        tokenizer.apply_chat_template(
            msg, tokenize=False, add_generation_prompt=False
        )
        for msg in examples["messages"]
    ]
    return tokenizer(texts, max_length=1024, truncation=True, padding=False)

tokenized_dataset = raw_dataset.map(
    preprocess_function, batched=True, remove_columns=raw_dataset.column_names
)

# 4. Training Arguments — resume_from_checkpoint handles optimizer/scheduler state
is_bf16 = torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen_graphmind_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=6,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    logging_steps=20,
    save_strategy="epoch",
    fp16=not is_bf16,
    bf16=is_bf16,
    optim="adamw_torch",
    seed=42,
    report_to="none",
    gradient_checkpointing=True,
    dataloader_num_workers=2,
)

# 5. Initialize Trainer & resume from checkpoint
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print(f"🚀 Resuming training from {CHECKPOINT}...")
trainer.train(resume_from_checkpoint=CHECKPOINT)  # restores step count, optimizer & scheduler

# 6. Save Adapter Weights
trainer.model.save_pretrained("/kaggle/working/qwen_graphmind_final")
tokenizer.save_pretrained("/kaggle/working/qwen_graphmind_final")
print("✅ Fine-tuning complete! Model saved to '/kaggle/working/qwen_graphmind_final'.")'''

In [ ]:
'''import os
import re
import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

# Free any cached memory before starting
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CHECKPOINT_PATH = "/kaggle/working/Bot-Detection/checkpoints/checkpoint-1500"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

is_bf16 = torch.cuda.is_bf16_supported()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if is_bf16 else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False
model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH, is_trainable=True)

# Dataset
jsonl_path = "/kaggle/working/Bot-Detection/data/qwen_combined_train.jsonl"
raw_dataset = load_dataset("json", data_files=jsonl_path, split="train")
grpo_dataset = raw_dataset.map(
    lambda ex: {"prompt": tokenizer.apply_chat_template(
        [m for m in ex["messages"] if m["role"] != "assistant"],
        tokenize=False, add_generation_prompt=True
    )},
    remove_columns=raw_dataset.column_names
).select(range(2000))

# ── Reward functions ──────────────────────────────────────────────────────

def think_tag_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        if "<think>" in c and "</think>" in c:
            score = 1.0
            think_content = re.findall(r'<think>(.*?)</think>', c, re.DOTALL)
            if think_content:
                length = len(think_content[0].strip())
                if length > 200: score += 0.5
                if length > 400: score += 0.5
            rewards.append(score)
        elif "<think>" in c:
            rewards.append(0.3)
        else:
            rewards.append(0.0)
    return rewards

def path_quality_reward(completions, **kwargs):
    rewards = []
    for c in completions:
        score = 0.0
        steps = re.findall(r'step \d+', c, re.IGNORECASE)
        score += min(len(steps) * 0.2, 0.6)
        if re.search(r'(total|overall).*score.*[\d.]+', c, re.IGNORECASE):
            score += 0.5
        if re.search(r'[\d.]+\s*[*x+]\s*[\d.]+', c):
            score += 0.3
        if len(c) < 100:
            score -= 0.5
        rewards.append(max(score, 0.0))
    return rewards

def diversity_reward(completions, **kwargs):
    if len(set(completions)) == 1:
        return [-0.3] * len(completions)
    return [0.0] * len(completions)

# ── Memory-safe config ────────────────────────────────────────────────────

training_args = GRPOConfig(
    output_dir="/kaggle/working/qwen_graphmind_grpo",
    learning_rate=1e-6,
    per_device_train_batch_size=1,      # down from 2
    gradient_accumulation_steps=4,      # keeps effective batch = 4
    num_generations=2,                  # down from 4 — biggest VRAM saving
    max_completion_length=256,          # down from 512
    max_steps=500,
    logging_steps=10,
    logging_first_step=True,
    save_strategy="steps",
    save_steps=100,
    fp16=not is_bf16,
    bf16=is_bf16,
    report_to="none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=42,
    generation_kwargs={
        "max_new_tokens": 256,          # matches max_completion_length
        "temperature": 0.9,
        "top_p": 0.95,
        "do_sample": True,
    },
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[think_tag_reward, path_quality_reward, diversity_reward],
    args=training_args,
    train_dataset=grpo_dataset,
)

print("🚀 Starting GRPO (memory-optimised)...")
trainer.train()

print("\n--- Training Summary ---")
for log in trainer.state.log_history:
    if "loss" in log:
        print(f"  step {log.get('step'):>4} | loss: {log.get('loss', 0):.4f} | reward: {log.get('reward', 'N/A')}")

save_dir = "/kaggle/working/qwen_graphmind_grpo_final"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\n✅ GRPO complete! Saved to '{save_dir}'.")'''